# Hybrid NER Labeling: Electra Model + Regex + Gazetteer

**Dataset:** `raw_data_2_pre.csv` — ~13,900 bài báo tiếng Việt

## Kiến trúc 3 tầng (Hybrid Pipeline)

```
┌─────────────────────────────────────────────────────────┐
│  Layer 1: Electra NER Model                             │
│  NlpHUST/ner-vietnamese-electra-base                    │
│  → PERSON, ORGANIZATION, LOCATION  (F1 ~0.92)          │
├─────────────────────────────────────────────────────────┤
│  Layer 2: Regex Patterns                                │
│  → MONEY, DATE, PERCENT  (cấu trúc rõ ràng)            │
├─────────────────────────────────────────────────────────┤
│  Layer 3: Gazetteer Dictionary                          │
│  → PRODUCT, EVENT, INDUSTRY                             │
│  → Bổ sung ORGANIZATION, LOCATION, PERSON mà model bỏ  │
│    sót (tên viết tắt, tên nước ngoài, tên riêng...)    │
└─────────────────────────────────────────────────────────┘
                        ↓
              Merge Engine (ưu tiên)
         Model > Regex > Gazetteer
                        ↓
                  BIO Output
```

**Chiến lược merge:**
1. Model Electra chạy trước → nhận diện PERSON, ORG, LOC dựa trên ngữ cảnh
2. Regex bổ sung → MONEY, DATE, PERCENT (model không cover)
3. Gazetteer bổ sung → PRODUCT, EVENT, INDUSTRY + patch cho entity model bỏ sót
4. Nếu span bị trùng: ưu tiên **Model > Regex > Gazetteer**

In [7]:
import pandas as pd
import re
import json
import time
import unicodedata
import warnings
from collections import Counter
from typing import List, Tuple, Dict, Optional

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline,
)

warnings.filterwarnings("ignore")

assert torch.cuda.is_available(), (
    "CUDA không khả dụng! Cài PyTorch GPU: "
    "pip install --force-reinstall torch torchvision torchaudio "
    "--index-url https://download.pytorch.org/whl/cu128"
)

DEVICE = 0
gpu_props = torch.cuda.get_device_properties(0)
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {gpu_props.total_memory / 1024**3:.1f} GB")
print(f"CUDA    : {torch.version.cuda}")

df = pd.read_csv("raw_data_2_pre.csv")
print(f"\nTổng bài báo: {len(df):,}")
print(f"Cột: {list(df.columns)}")
print(f"Missing:\n{df.isnull().sum()}")
df.head(2)

PyTorch : 2.10.0+cu128
GPU     : NVIDIA GeForce RTX 3070 Laptop GPU
VRAM    : 8.0 GB
CUDA    : 12.8

Tổng bài báo: 13,906
Cột: ['id', 'title', 'content', 'source', 'date']
Missing:
id         0
title      0
content    0
source     0
date       0
dtype: int64


,id,title,content,source,date
0,1,Motorola chuẩn bị ra mắt phiên bản điện thoại ...,Thiết bị mới đã xuất hiện trên nền tảng Geekbe...,vietnamnet,2024-09-24
1,2,"2 tỷ phú quốc tế 'xông đất', hứa đưa Bình Định...","Ngày 16/1, tỉnh Bình Định đã chào đón hai nhà ...",vietnamnet,2025-01-16


## Text Normalization

Chuẩn hoá text trước khi NER: Unicode NFC, whitespace, ký tự đặc biệt → tránh miss match.

In [8]:
_NORMALIZE_MAP = {
    "\u00a0": " ",   # non-breaking space
    "\u200b": "",    # zero-width space
    "\u200c": "",    # zero-width non-joiner
    "\u200d": "",    # zero-width joiner
    "\ufeff": "",    # BOM
    "\u2013": "-",   # en-dash → hyphen
    "\u2014": "-",   # em-dash → hyphen
    "\u2018": "'",   # left single quote
    "\u2019": "'",   # right single quote
    "\u201c": '"',   # left double quote
    "\u201d": '"',   # right double quote
    "\u2026": "...", # ellipsis
    "\u202f": " ",   # narrow no-break space
}
_NORM_RE = re.compile("|".join(re.escape(k) for k in _NORMALIZE_MAP))
_MULTI_SPACE = re.compile(r"[ \t]{2,}")


def normalize_text(text: str) -> str:
    """
    Chuẩn hoá text:
    1. Unicode NFC (tổ hợp dấu tiếng Việt về dạng chuẩn)
    2. Thay thế ký tự đặc biệt (smart quotes, dashes, NBSP...)
    3. Gộp khoảng trắng thừa
    4. Strip đầu cuối
    """
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = _NORM_RE.sub(lambda m: _NORMALIZE_MAP[m.group()], text)
    text = _MULTI_SPACE.sub(" ", text)
    return text.strip()


# Test normalization
test_cases = [
    "Ngày\u00a016/1,\u2003ông\u200bRoland",            # NBSP + em-space + ZWSP
    "GDP tăng\u2013trưởng 6,5%",                        # en-dash
    "\u201cChuyển đổi số\u201d nâng cao",                # smart quotes
    "Giá  tương   đương   12  triệu\u00a0đồng",         # multi-space + NBSP
    "Hà\u0300 No\u0323i",                                # NFD decomposed: Hà Nội
]
for tc in test_cases:
    norm = normalize_text(tc)
    changed = tc != norm
    print(f"{'→' if changed else ' '} '{tc}' → '{norm}'")

→ 'Ngày 16/1, ông​Roland' → 'Ngày 16/1, ôngRoland'
→ 'GDP tăng–trưởng 6,5%' → 'GDP tăng-trưởng 6,5%'
→ '“Chuyển đổi số” nâng cao' → '"Chuyển đổi số" nâng cao'
→ 'Giá  tương   đương   12  triệu đồng' → 'Giá tương đương 12 triệu đồng'
→ 'Hà̀ Nọi' → 'Hà̀ Nọi'


## Layer 1: Electra NER Model

`NlpHUST/ner-vietnamese-electra-base` — Fine-tuned trên VLSP 2018, F1 = 0.92

Nhận diện: **PERSON** (F1=0.97), **ORGANIZATION** (F1=0.88), **LOCATION** (F1=0.94)

In [9]:
MODEL_NAME = "NlpHUST/ner-vietnamese-electra-base"

print(f"Loading model: {MODEL_NAME} ...")
ner_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
ner_model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
ner_model = ner_model.to("cuda")
ner_model.eval()

# batch_size=16 tối ưu cho RTX 3070 8GB, model ~110M params chỉ tốn ~0.5GB VRAM
ner_pipeline = pipeline(
    "ner",
    model=ner_model,
    tokenizer=ner_tokenizer,
    aggregation_strategy="simple",
    device=DEVICE,
    batch_size=16,
)

ELECTRA_LABEL_MAP = {
    "PER": "PERSON",
    "PERSON": "PERSON",
    "ORG": "ORGANIZATION",
    "ORGANIZATION": "ORGANIZATION",
    "LOC": "LOCATION",
    "LOCATION": "LOCATION",
    "MISC": "MISCELLANEOUS",
    "MISCELLANEOUS": "MISCELLANEOUS",
}

vram_used = torch.cuda.memory_allocated(0) / 1024**2
print(f"Model loaded on GPU ({vram_used:.0f} MB VRAM)")
print(f"Label map: {ner_model.config.id2label}")
print(f"Batch size: 16")

Loading model: NlpHUST/ner-vietnamese-electra-base ...


Device set to use cuda:0


Model loaded on GPU (518 MB VRAM)
Label map: {0: 'B-LOCATION', 1: 'B-MISCELLANEOUS', 2: 'B-ORGANIZATION', 3: 'B-PERSON', 4: 'I-LOCATION', 5: 'I-MISCELLANEOUS', 6: 'I-ORGANIZATION', 7: 'I-PERSON', 8: 'O'}
Batch size: 16


In [10]:
@torch.no_grad()
def electra_ner(text: str, min_score: float = 0.70) -> List[Tuple[int, int, str, float]]:
    """
    Chạy Electra NER trên 1 text (dùng cho test/debug).
    Batch processing dùng batch_electra_ner() ở cell sau.
    """
    if not isinstance(text, str) or not text.strip():
        return []

    MAX_CHARS = 1500
    chunks = []
    if len(text) <= MAX_CHARS:
        chunks = [(text, 0)]
    else:
        sentences = re.split(r'(?<=[.!?])\s+', text)
        current_chunk, current_offset = "", 0
        for sent in sentences:
            if len(current_chunk) + len(sent) + 1 > MAX_CHARS and current_chunk:
                chunks.append((current_chunk, current_offset))
                current_offset += len(current_chunk) + 1
                current_chunk = sent
            else:
                current_chunk = (current_chunk + " " + sent) if current_chunk else sent
        if current_chunk:
            chunks.append((current_chunk, current_offset))

    results = []
    for chunk_text, offset in chunks:
        try:
            ner_results = ner_pipeline(chunk_text)
        except Exception:
            continue

        for ent in ner_results:
            mapped = ELECTRA_LABEL_MAP.get(ent["entity_group"], ent["entity_group"])
            if ent["score"] < min_score or mapped == "MISCELLANEOUS":
                continue
            results.append((ent["start"] + offset, ent["end"] + offset, mapped, ent["score"]))

    return results


# Test — chạy trên GPU
test_text = (
    "Ngày 16/1, ông Roland Staub - Chủ tịch Quỹ đầu tư Finance Suisse và "
    "ông Timur Mohamed - Chủ tịch công ty Palmer Johnson đến thăm Bình Định."
)
print(f"Input: {test_text}\n")
print(f"Electra NER results (on {torch.cuda.get_device_name(0)}):")
for start, end, etype, score in electra_ner(test_text):
    print(f"  [{start:3d}:{end:3d}] {etype:15s} ({score:.3f}) │ '{test_text[start:end]}'")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Input: Ngày 16/1, ông Roland Staub - Chủ tịch Quỹ đầu tư Finance Suisse và ông Timur Mohamed - Chủ tịch công ty Palmer Johnson đến thăm Bình Định.

Electra NER results (on NVIDIA GeForce RTX 3070 Laptop GPU):
  [ 15: 27] PERSON          (0.999) │ 'Roland Staub'
  [ 39: 49] ORGANIZATION    (0.952) │ 'Quỹ đầu tư'
  [ 50: 64] ORGANIZATION    (0.965) │ 'Finance Suisse'
  [ 72: 85] PERSON          (0.999) │ 'Timur Mohamed'
  [105:119] ORGANIZATION    (0.908) │ 'Palmer Johnson'
  [129:138] LOCATION        (0.999) │ 'Bình Định'


## Layer 2: Regex Patterns

Nhận diện entity có cấu trúc rõ ràng: **MONEY**, **DATE**, **PERCENT**

In [11]:
CURRENCY = r'(?:USD|VND|VNĐ|đồng|đô\s*la|CHF|EUR|JPY|CNY|nhân\s+dân\s+tệ)'
UNIT     = r'(?:triệu|tỷ|nghìn|ngàn|trăm)'
NUM      = r'\d[\d.,]*'

MONEY_PATTERNS = [
    # "12 triệu đồng", "1,66 tỷ USD", "500 nghìn VNĐ"
    re.compile(rf'\b{NUM}\s*{UNIT}\s*{CURRENCY}\b', re.IGNORECASE),
    # "1000 USD", "700 đồng" (số + tiền tệ, không qua đơn vị)
    re.compile(rf'\b{NUM}\s*{CURRENCY}\b', re.IGNORECASE),
    # "$1000", "$ 1,000"
    re.compile(rf'\$\s*{NUM}\b'),
    # "50 triệu", "1,2 tỷ" (số + đơn vị, không có currency nhưng ngữ cảnh tiền)
    re.compile(rf'\b{NUM}\s*{UNIT}(?=\s|,|\.|\)|$)', re.IGNORECASE),
]

DATE_PATTERNS = [
    # "ngày 16/1", "Ngày 10/9/2024"
    re.compile(r'\b[Nn]gày\s+\d{1,2}/\d{1,2}(?:/\d{2,4})?\b'),
    # "tháng 2/2026", "tháng 11"
    re.compile(r'\b[Tt]háng\s+\d{1,2}(?:/\d{2,4})?\b'),
    # "năm 2024"
    re.compile(r'\b[Nn]ăm\s+\d{4}\b'),
    # ISO: "2024-09-24"
    re.compile(r'\b\d{4}-\d{2}-\d{2}\b'),
    # "31/12/2024" (standalone, không phải phần của text khác)
    re.compile(r'(?<!\w)\d{1,2}/\d{1,2}/\d{2,4}(?!\w)'),
    # "quý IV/2024"
    re.compile(r'\b[Qq]uý\s+[IViv]+/\d{4}\b'),
    # "đầu năm 2024", "cuối năm 2025", "mùa hè năm 2024"
    re.compile(
        r'\b(?:đầu|cuối|giữa|mùa\s+hè|mùa\s+thu|mùa\s+đông|mùa\s+xuân)\s+năm\s+\d{4}\b',
        re.IGNORECASE
    ),
    # "giai đoạn 2025-2028", "giai đoạn 2021 - 2025"
    re.compile(r'\bgiai\s+đoạn\s+\d{4}\s*[-–]\s*\d{4}\b', re.IGNORECASE),
]

PERCENT_PATTERNS = [
    # "6,5%", "30%", "0,3%/năm", "16,4%"
    re.compile(r'(?<!\w)\d[\d.,]*\s*%(?:/năm)?(?!\w)'),
]


def regex_ner(text: str) -> List[Tuple[int, int, str]]:
    """
    Tìm MONEY, DATE, PERCENT bằng regex.
    Trả về list of (start, end, entity_type).
    Loại bỏ duplicate spans (cùng start-end giữ pattern đầu tiên match).
    """
    if not isinstance(text, str):
        return []

    spans = []
    seen = set()
    for patterns, etype in [
        (MONEY_PATTERNS, "MONEY"),
        (DATE_PATTERNS, "DATE"),
        (PERCENT_PATTERNS, "PERCENT"),
    ]:
        for pat in patterns:
            for m in pat.finditer(text):
                key = (m.start(), m.end())
                if key not in seen:
                    seen.add(key)
                    spans.append((m.start(), m.end(), etype))

    return spans


# Test
test_text = "Ngày 16/1, GDP tăng 6,5% trong năm 2024. Giá 1000 USD, tương đương 12 triệu đồng."
print(f"Input: {test_text}\n")
for s, e, t in regex_ner(test_text):
    print(f"  [{s:3d}:{e:3d}] {t:10s} │ '{test_text[s:e]}'")

# Test edge cases
print("\n--- Edge cases ---")
for tc in [
    "tăng 0,3%/năm lên 4,6%.",
    "giai đoạn 2025 - 2028",
    "31/12/2024 là ngày cuối cùng",
    "khoảng 850.000m3 nước",
    "$1,000 hoặc 500 nghìn VNĐ",
]:
    results = regex_ner(tc)
    print(f"  '{tc}'")
    for s, e, t in results:
        print(f"    [{s:3d}:{e:3d}] {t:10s} │ '{tc[s:e]}'")

Input: Ngày 16/1, GDP tăng 6,5% trong năm 2024. Giá 1000 USD, tương đương 12 triệu đồng.

  [ 67: 80] MONEY      │ '12 triệu đồng'
  [ 45: 53] MONEY      │ '1000 USD'
  [ 67: 75] MONEY      │ '12 triệu'
  [  0:  9] DATE       │ 'Ngày 16/1'
  [ 31: 39] DATE       │ 'năm 2024'
  [ 20: 24] PERCENT    │ '6,5%'

--- Edge cases ---
  'tăng 0,3%/năm lên 4,6%.'
    [  5: 13] PERCENT    │ '0,3%/năm'
    [ 18: 22] PERCENT    │ '4,6%'
  'giai đoạn 2025 - 2028'
    [  0: 21] DATE       │ 'giai đoạn 2025 - 2028'
  '31/12/2024 là ngày cuối cùng'
    [  0: 10] DATE       │ '31/12/2024'
  'khoảng 850.000m3 nước'
  '$1,000 hoặc 500 nghìn VNĐ'
    [ 12: 25] MONEY      │ '500 nghìn VNĐ'
    [  0:  6] MONEY      │ '$1,000'
    [ 12: 21] MONEY      │ '500 nghìn'


## Layer 3: Gazetteer Dictionary

Vai trò kép:
- **Nhận diện entity model không cover:** PRODUCT, EVENT, INDUSTRY
- **Bổ sung entity model bỏ sót:** tên viết tắt (ACB, BIDV), tên nước ngoài (Elon Musk), sản phẩm (iPhone 17)...

In [12]:
# === GAZETTEER: entity mà model KHÔNG cover ===

GAZ_PRODUCT = [
    "iPhone", "iPhone 15", "iPhone 17", "iPhone 17 Pro Max", "iPhone Air",
    "Razr", "Razr 50s", "Razr 50", "Razr 50 Ultra", "Razr 2024", "Razr Plus 2024",
    "Galaxy", "Pixel",
    "MediaTek 7300", "Snapdragon", "AirPods", "AirPods Pro", "AirPods Pro 3",
    "Apple Watch", "Apple Watch Series 11", "Apple Watch Ultra",
    "Apple Watch Ultra 3", "Apple Watch SE 3", "HomePod", "HomeHub",
    "ChatGPT", "Gemini", "Siri", "Alexa", "Copilot",
    "Android", "Android 14", "iOS", "iOS 26",
    "watchOS 26", "iPadOS 26", "macOS Tahoe",
    "Apple Intelligence", "Apple Foundation Models",
    "App Store", "Google Play", "Open Banking", "OneBank", "OneBank 365+",
    "CheckVN", "VDAPES", "vTravel",
    "5G Advanced", "5G SA", "5G NSA", "LTE Advanced", "MIMO",
]

GAZ_EVENT = [
    "Gaming On TikTok Hanoi Summit 2025",
    "HR Asia Awards 2025", "HR Asia Awards",
    "WWDC", "WWDC 2024",
    "Hội nghị các nhà phát triển toàn cầu",
    "SEA Games", "ASIAD", "Olympic", "World Cup",
    "AFF Cup", "Champions League",
    "Tết Nguyên Đán", "Tết Ất Tỵ", "vía Thần Tài",
    "Make in Viet Nam", "Awe Dropping",
    "COP26", "COP27", "COP28", "COP29",
    "Ngày hội An toàn", "EHS Day",
    "LUXignite",
]

GAZ_INDUSTRY = [
    "bất động sản", "ngân hàng", "tài chính", "chứng khoán",
    "công nghệ", "viễn thông", "thương mại điện tử",
    "du lịch", "hàng không", "logistics", "vận tải",
    "nông nghiệp", "công nghiệp", "y tế", "giáo dục",
    "xây dựng", "năng lượng", "dầu khí", "bảo hiểm",
    "game", "điện ảnh", "thời trang", "ẩm thực",
    "nông nghiệp công nghệ cao", "thương mại điện tử xuyên biên giới",
    "công nghiệp game", "ngành game", "phòng cháy chữa cháy",
]

GAZ_ORGANIZATION_PATCH = [
    "BIDV", "ACB", "SHB", "OCB", "MSB", "VIB", "NCB", "KLB",
    "HDBank", "LPBank", "TPBank", "VPBank", "ABBank", "BVBank",
    "SeABank", "PGBank", "VietBank", "Saigonbank", "KienlongBank", "Bac A Bank",
    "Viet A Bank", "Nam A Bank",
    "HoSE", "HNX", "UPCOM", "NAPAS",
    "NHNN", "UBND", "SECO",
    "WHO", "UNESCO", "ASEAN", "EU", "NATO", "IMF", "WTO", "APEC", "3GPP",
    "Finance Suisse", "Palmer Johnson", "Bloomberg Intelligence",
    "Sensor Tower", "Statista", "LightReading", "The Verge", "HR Asia",
    "Phát Đạt Corporation",
    "Cục Thương mại Điện tử và Kinh tế số",
    "Cục Quản lý và Phát triển thị trường trong nước",
    "Cục Công nghiệp CNTT",
    "Đại học Bách Khoa Hà Nội",
    "Tổng cục Kinh tế Liên bang Thụy Sĩ",
]

GAZ_LOCATION_PATCH = [
    "Đông Nam Á", "Châu Á", "Tây Bắc", "Đông Bắc",
    "Thung lũng Silicon", "Silicon Valley",
    "Biển Đông", "Hoàng Sa", "Trường Sa",
    "chợ đầu mối Bình Điền",
    "TP.HCM", "TP HCM",
]

GAZ_PERSON_PATCH = [
    "Mark Zuckerberg", "Zuckerberg", "Steve Jobs", "Tim Cook",
    "Elon Musk", "Jeff Bezos", "Sundar Pichai",
    "Andy Jassy", "Børje Ekholm", "Dan Ives",
    "Joe Rogan", "Rogan",
    "Tomasz Noetzel", "Tse King Kiu",
    "Park Hang Seo", "Park Hang-seo",
]

# ─── Build sorted gazetteer (PRE-SORT 1 lần, dài nhất trước) ───
# Mỗi entry: (normalized_name, original_name, entity_type, case_sensitive)
# Sort theo len giảm dần → match dài nhất trước, tránh "iPhone" nuốt "iPhone 17"

def _build_sorted_gazetteer(raw_gazetteers: dict) -> list:
    entries = []
    for etype, (gaz_list, case_sensitive) in raw_gazetteers.items():
        for name in gaz_list:
            norm = normalize_text(name)
            entries.append((norm, name, etype, case_sensitive))
    entries.sort(key=lambda x: -len(x[0]))
    return entries

_RAW_GAZETTEERS = {
    "PRODUCT":      (GAZ_PRODUCT, True),
    "EVENT":        (GAZ_EVENT, True),
    "INDUSTRY":     (GAZ_INDUSTRY, False),
    "ORGANIZATION": (GAZ_ORGANIZATION_PATCH, True),
    "LOCATION":     (GAZ_LOCATION_PATCH, True),
    "PERSON":       (GAZ_PERSON_PATCH, True),
}

SORTED_GAZETTEER = _build_sorted_gazetteer(_RAW_GAZETTEERS)

total = len(SORTED_GAZETTEER)
by_type = Counter(e[2] for e in SORTED_GAZETTEER)
print(f"Gazetteer tổng: {total} entries (pre-sorted by length desc)")
for etype, cnt in by_type.most_common():
    cs = next(e[3] for e in SORTED_GAZETTEER if e[2] == etype)
    print(f"  {etype:15s}: {cnt:3d} entries  (case_sensitive={cs})")
print(f"\nTop-5 dài nhất:")
for norm, orig, etype, cs in SORTED_GAZETTEER[:5]:
    print(f"  [{len(norm):3d} chars] {etype:15s} │ '{orig}'")

Gazetteer tổng: 184 entries (pre-sorted by length desc)
  ORGANIZATION   :  52 entries  (case_sensitive=True)
  PRODUCT        :  52 entries  (case_sensitive=True)
  INDUSTRY       :  28 entries  (case_sensitive=False)
  EVENT          :  24 entries  (case_sensitive=True)
  PERSON         :  16 entries  (case_sensitive=True)
  LOCATION       :  12 entries  (case_sensitive=True)

Top-5 dài nhất:
  [ 47 chars] ORGANIZATION    │ 'Cục Quản lý và Phát triển thị trường trong nước'
  [ 36 chars] EVENT           │ 'Hội nghị các nhà phát triển toàn cầu'
  [ 36 chars] ORGANIZATION    │ 'Cục Thương mại Điện tử và Kinh tế số'
  [ 34 chars] EVENT           │ 'Gaming On TikTok Hanoi Summit 2025'
  [ 34 chars] INDUSTRY        │ 'thương mại điện tử xuyên biên giới'


In [13]:
_VI_WORD_CHARS = set("aăâáắấàằầảẳẩãẵẫạặậ"
                     "eêéếèềẻểẽễẹệ"
                     "iíìỉĩị"
                     "oôơóốớòồờỏổởõỗỡọộợ"
                     "uưúứùừủửũữụự"
                     "yýỳỷỹỵ"
                     "đ"
                     "AĂÂÁẮẤÀẰẦẢẲẨÃẴẪẠẶẬ"
                     "EÊÉẾÈỀẺỂẼỄẸỆ"
                     "IÍÌỈĨỊ"
                     "OÔƠÓỐỚÒỒỜỎỔỞÕỖỠỌỘỢ"
                     "UƯÚỨÙỪỦỬŨỮỤỰ"
                     "YÝỲỶỸỴ"
                     "Đ")


def _is_word_char(ch: str) -> bool:
    """Kiểm tra ký tự có phải word character (bao gồm tiếng Việt)."""
    return ch.isalnum() or ch in _VI_WORD_CHARS


def _check_boundary(text: str, start: int, end: int) -> bool:
    """Kiểm tra word boundary cho tiếng Việt (regex \\b không hiểu dấu)."""
    before_ok = (start == 0 or not _is_word_char(text[start - 1]))
    after_ok = (end >= len(text) or not _is_word_char(text[end]))
    return before_ok and after_ok


def gazetteer_ner(text: str) -> List[Tuple[int, int, str]]:
    """
    Tìm entity bằng gazetteer pre-sorted.
    - Đã sort dài→ngắn: match dài nhất trước
    - Dùng occupied set: không cho span ngắn đè lên span dài đã match
    - Boundary check hỗ trợ tiếng Việt (dấu, unicode)
    """
    if not isinstance(text, str) or not text.strip():
        return []

    text_lower = text.lower()
    spans = []
    occupied = set()

    for norm_name, orig_name, etype, case_sensitive in SORTED_GAZETTEER:
        search_text = text if case_sensitive else text_lower
        search_name = norm_name if case_sensitive else norm_name.lower()

        pos = 0
        while True:
            idx = search_text.find(search_name, pos)
            if idx == -1:
                break
            end = idx + len(search_name)

            if _check_boundary(text, idx, end):
                char_range = set(range(idx, end))
                if not char_range & occupied:
                    spans.append((idx, end, etype))
                    occupied |= char_range

            pos = idx + 1

    spans.sort(key=lambda x: x[0])
    return spans


# Test
test_text = "Apple ra mắt iPhone 17 tại HR Asia Awards 2025, ngành công nghệ rất sôi động."
print(f"Input: {test_text}\n")
for s, e, t in gazetteer_ner(test_text):
    print(f"  [{s:3d}:{e:3d}] {t:15s} │ '{test_text[s:e]}'")

# Edge case: no more duplicates (iPhone 17 should NOT also match iPhone separately)
print("\n--- Overlap test (iPhone 17 vs iPhone) ---")
for s, e, t in gazetteer_ner("Tôi mua iPhone 17 và AirPods Pro 3."):
    print(f"  [{s:3d}:{e:3d}] {t:15s} │ match")

Input: Apple ra mắt iPhone 17 tại HR Asia Awards 2025, ngành công nghệ rất sôi động.

  [ 13: 22] PRODUCT         │ 'iPhone 17'
  [ 27: 46] EVENT           │ 'HR Asia Awards 2025'
  [ 54: 63] INDUSTRY        │ 'công nghệ'

--- Overlap test (iPhone 17 vs iPhone) ---
  [  8: 17] PRODUCT         │ match
  [ 21: 34] PRODUCT         │ match


## Merge Engine + Tokenizer + BIO Converter

Kết hợp 3 tầng, xử lý overlap, convert sang BIO format.

**Priority:** Model (score cao, ngữ cảnh) → Regex (cấu trúc) → Gazetteer (bổ sung)

In [14]:
PRIORITY_MODEL = 3
PRIORITY_REGEX = 2
PRIORITY_GAZ   = 1


def resolve_overlaps(
    spans: List[Tuple[int, int, str, int]]
) -> List[Tuple[int, int, str]]:
    """
    Giải quyết overlap giữa các span.
    Input: list of (start, end, entity_type, priority)
    Output: list of (start, end, entity_type) không trùng lấn.

    Chiến lược:
      1. Priority cao hơn thắng
      2. Cùng priority → span dài hơn thắng
      3. Cùng length → span xuất hiện trước thắng
    """
    spans_sorted = sorted(spans, key=lambda x: (-x[3], -(x[1] - x[0]), x[0]))

    resolved = []
    occupied = set()
    for start, end, etype, _ in spans_sorted:
        char_range = set(range(start, end))
        if not char_range & occupied:
            resolved.append((start, end, etype))
            occupied |= char_range

    resolved.sort(key=lambda x: x[0])
    return resolved


def hybrid_ner(text: str) -> List[Tuple[int, int, str]]:
    """
    Pipeline NER hybrid: normalize → Model + Regex + Gazetteer → merge.
    Text được normalize trước khi chạy qua 3 layers.
    """
    text = normalize_text(text)
    if not text:
        return []

    all_spans = []

    for start, end, etype, score in electra_ner(text):
        all_spans.append((start, end, etype, PRIORITY_MODEL))

    for start, end, etype in regex_ner(text):
        all_spans.append((start, end, etype, PRIORITY_REGEX))

    for start, end, etype in gazetteer_ner(text):
        all_spans.append((start, end, etype, PRIORITY_GAZ))

    return resolve_overlaps(all_spans)


# === Tokenizer with offsets ===

def tokenize_with_offsets(text: str) -> List[Tuple[str, int, int]]:
    """
    Tách token theo khoảng trắng + tách dấu câu dính.
    Trả về (token, start_char, end_char).
    """
    if not isinstance(text, str) or not text.strip():
        return []

    result = []
    raw_tokens = text.split()
    pos = 0

    for raw in raw_tokens:
        idx = text.find(raw, pos)
        if idx == -1:
            idx = pos
        offset = idx

        # Leading punctuation
        i = 0
        while i < len(raw) and raw[i] in '([{"\'"«»\u201c\u201d\u2018\u2019':
            result.append((raw[i], offset + i, offset + i + 1))
            i += 1

        # Trailing punctuation
        j = len(raw)
        trailing = []
        while j > i and raw[j - 1] in '.,;:!?)]}"\'"»\u201d\u2019\u2026':
            if raw[j - 1] in '.,' and j - 2 >= i and raw[j - 2].isdigit():
                break
            trailing.append((raw[j - 1], offset + j - 1, offset + j))
            j -= 1

        if j > i:
            result.append((raw[i:j], offset + i, offset + j))

        for t in reversed(trailing):
            result.append(t)

        pos = idx + len(raw)

    return result


# === Spans → BIO labels ===

def spans_to_bio(
    text: str,
    entity_spans: List[Tuple[int, int, str]]
) -> List[Tuple[str, str]]:
    """
    Chuyển character-level entity spans thành token-level BIO labels.
    Trả về list of (token, label).
    """
    token_data = tokenize_with_offsets(text)
    result = []

    for token, t_start, t_end in token_data:
        label = "O"
        for e_start, e_end, e_type in entity_spans:
            if t_start >= e_start and t_end <= e_end:
                label = f"B-{e_type}" if t_start == e_start else f"I-{e_type}"
                break
        result.append((token, label))

    return result


def hybrid_ner_bio(text: str) -> List[Tuple[str, str]]:
    """Full pipeline: normalize → hybrid NER → BIO labels."""
    text = normalize_text(text)
    if not text:
        return []
    spans = hybrid_ner(text)
    return spans_to_bio(text, spans)


print("Merge engine ready.")

Merge engine ready.


## Test Hybrid Pipeline

So sánh kết quả từng layer và kết quả merged.

In [15]:
def show_ner_comparison(text: str):
    """Hiển thị kết quả từng layer + merged."""
    print(f"{'='*70}")
    print(f"INPUT: {text[:100]}{'...' if len(text)>100 else ''}")
    print(f"{'='*70}")

    # Layer 1: Model
    model_spans = electra_ner(text)
    print(f"\n[Layer 1 - Electra Model] ({len(model_spans)} entities)")
    for s, e, t, sc in model_spans:
        print(f"  {t:15s} ({sc:.2f}) │ '{text[s:e]}'")

    # Layer 2: Regex
    regex_spans = regex_ner(text)
    print(f"\n[Layer 2 - Regex] ({len(regex_spans)} entities)")
    for s, e, t in regex_spans:
        print(f"  {t:15s}        │ '{text[s:e]}'")

    # Layer 3: Gazetteer
    gaz_spans = gazetteer_ner(text)
    print(f"\n[Layer 3 - Gazetteer] ({len(gaz_spans)} entities)")
    for s, e, t in gaz_spans:
        print(f"  {t:15s}        │ '{text[s:e]}'")

    # Merged
    merged = hybrid_ner(text)
    print(f"\n[MERGED] ({len(merged)} entities)")
    for s, e, t in merged:
        print(f"  {t:15s}        │ '{text[s:e]}'")

    # BIO output
    bio = hybrid_ner_bio(text)
    print(f"\n[BIO Output]")
    for token, label in bio:
        if label != "O":
            print(f"  {token:25s} {label}")

    print()


# Test 1: Bài đầu tiên
show_ner_comparison(str(df.iloc[0]["title"]))

# Test 2: Câu phức tạp
show_ner_comparison(
    "Ngày 16/1, ông Roland Staub đến Bình Định, cam kết hỗ trợ 50 triệu USD "
    "cho ngành du lịch, tăng 6,5% so với năm 2024."
)

# Test 3: Công nghệ
show_ner_comparison(
    "Apple bắt tay Google: Gemini chính thức trở thành bộ não mới cho Siri, "
    "thỏa thuận trị giá 1 tỷ USD tại WWDC 2024."
)

INPUT: Motorola chuẩn bị ra mắt phiên bản điện thoại nắp gập Razr giá rẻ 12 triệu đồng

[Layer 1 - Electra Model] (1 entities)
  ORGANIZATION    (1.00) │ 'Motorola'

[Layer 2 - Regex] (2 entities)
  MONEY                  │ '12 triệu đồng'
  MONEY                  │ '12 triệu'

[Layer 3 - Gazetteer] (1 entities)
  PRODUCT                │ 'Razr'

[MERGED] (3 entities)
  ORGANIZATION           │ 'Motorola'
  PRODUCT                │ 'Razr'
  MONEY                  │ '12 triệu đồng'

[BIO Output]
  Motorola                  B-ORGANIZATION
  Razr                      B-PRODUCT
  12                        B-MONEY
  triệu                     I-MONEY
  đồng                      I-MONEY

INPUT: Ngày 16/1, ông Roland Staub đến Bình Định, cam kết hỗ trợ 50 triệu USD cho ngành du lịch, tăng 6,5% ...

[Layer 1 - Electra Model] (2 entities)
  PERSON          (1.00) │ 'Roland Staub'
  LOCATION        (1.00) │ 'Bình Định'

[Layer 2 - Regex] (5 entities)
  MONEY                  │ '50 triệu USD'
  MO

## Áp dụng lên toàn bộ dataset

Batch processing với progress tracking. Content dài được cắt chunk để model xử lý hiệu quả.

In [17]:
def bio_to_string(bio: List[Tuple[str, str]]) -> str:
    return "\n".join(f"{token} {label}" for token, label in bio)


def batch_electra_ner(
    texts: List[str], min_score: float = 0.70
) -> List[List[Tuple[int, int, str, float]]]:
    """
    Batch inference: chạy Electra NER trên nhiều text cùng lúc.
    GPU sẽ xử lý batch_size=16 text song song → nhanh hơn ~10x so với từng text.
    """
    all_results = [[] for _ in texts]
    chunk_map = []  # (text_idx, chunk_text, offset)

    for i, text in enumerate(texts):
        if not isinstance(text, str) or not text.strip():
            continue
        MAX_CHARS = 1500
        if len(text) <= MAX_CHARS:
            chunk_map.append((i, text, 0))
        else:
            sentences = re.split(r'(?<=[.!?])\s+', text)
            current_chunk, current_offset = "", 0
            for sent in sentences:
                if len(current_chunk) + len(sent) + 1 > MAX_CHARS and current_chunk:
                    chunk_map.append((i, current_chunk, current_offset))
                    current_offset += len(current_chunk) + 1
                    current_chunk = sent
                else:
                    current_chunk = (current_chunk + " " + sent) if current_chunk else sent
            if current_chunk:
                chunk_map.append((i, current_chunk, current_offset))

    if not chunk_map:
        return all_results

    chunk_texts = [c[1] for c in chunk_map]

    try:
        pipe_results = ner_pipeline(chunk_texts)
    except Exception:
        pipe_results = [[] for _ in chunk_texts]

    if chunk_texts and not isinstance(pipe_results[0], list):
        pipe_results = [pipe_results]

    for (text_idx, _, offset), ents in zip(chunk_map, pipe_results):
        for ent in ents:
            raw_label = ent["entity_group"]
            mapped = ELECTRA_LABEL_MAP.get(raw_label, raw_label)
            if ent["score"] < min_score or mapped == "MISCELLANEOUS":
                continue
            all_results[text_idx].append(
                (ent["start"] + offset, ent["end"] + offset, mapped, ent["score"])
            )

    return all_results


def process_column_gpu(series: pd.Series, col_name: str, batch_sz: int = 64) -> pd.Series:
    """
    Batch processing trên GPU.
    - Gom batch_sz texts → chạy Electra 1 lần cho cả batch
    - Regex + Gazetteer vẫn chạy per-text (rất nhanh, ko cần GPU)
    """
    total = len(series)
    texts = [normalize_text(str(t)) if pd.notna(t) else "" for t in series]
    results = [""] * total
    start_time = time.time()

    for batch_start in range(0, total, batch_sz):
        batch_end = min(batch_start + batch_sz, total)
        batch_texts = texts[batch_start:batch_end]

        # Layer 1: Electra batch inference on GPU
        model_results = batch_electra_ner(batch_texts)

        for j, text in enumerate(batch_texts):
            idx = batch_start + j
            all_spans = []

            # Model spans
            for s, e, t, sc in model_results[j]:
                all_spans.append((s, e, t, PRIORITY_MODEL))

            # Layer 2+3: Regex + Gazetteer (CPU, rất nhanh)
            for s, e, t in regex_ner(text):
                all_spans.append((s, e, t, PRIORITY_REGEX))
            for s, e, t in gazetteer_ner(text):
                all_spans.append((s, e, t, PRIORITY_GAZ))

            merged = resolve_overlaps(all_spans)
            bio = spans_to_bio(text, merged)
            results[idx] = bio_to_string(bio)

        done = batch_end
        elapsed = time.time() - start_time
        speed = done / elapsed if elapsed > 0 else 0
        eta = (total - done) / speed if speed > 0 else 0
        vram = torch.cuda.memory_allocated(0) / 1024**2

        if done % 500 < batch_sz or done == total:
            print(f"  [{col_name}] {done:>6,}/{total:,} "
                  f"({done/total*100:5.1f}%) "
                  f"| {elapsed:.0f}s | ~{eta:.0f}s left | "
                  f"{speed:.1f} rows/s | VRAM {vram:.0f}MB")

    return pd.Series(results, index=series.index)


# === Xử lý TITLE trên GPU ===
torch.cuda.empty_cache()
print("=" * 60)
print("PROCESSING TITLES (GPU batch inference)")
print("=" * 60)
t0 = time.time()
df["title_bio"] = process_column_gpu(df["title"], "title", batch_sz=64)
title_time = time.time() - t0
print(f"\nTitle done: {title_time:.1f}s\n")

PROCESSING TITLES (GPU batch inference)
  [title]    512/13,906 (  3.7%) | 1s | ~21s left | 646.9 rows/s | VRAM 525MB
  [title]  1,024/13,906 (  7.4%) | 2s | ~21s left | 602.3 rows/s | VRAM 525MB
  [title]  1,536/13,906 ( 11.0%) | 3s | ~21s left | 589.8 rows/s | VRAM 525MB
  [title]  2,048/13,906 ( 14.7%) | 4s | ~21s left | 575.5 rows/s | VRAM 525MB
  [title]  2,560/13,906 ( 18.4%) | 5s | ~20s left | 567.7 rows/s | VRAM 525MB
  [title]  3,008/13,906 ( 21.6%) | 5s | ~19s left | 559.0 rows/s | VRAM 525MB
  [title]  3,520/13,906 ( 25.3%) | 6s | ~19s left | 548.6 rows/s | VRAM 525MB
  [title]  4,032/13,906 ( 29.0%) | 8s | ~19s left | 530.1 rows/s | VRAM 525MB
  [title]  4,544/13,906 ( 32.7%) | 9s | ~19s left | 503.2 rows/s | VRAM 525MB
  [title]  5,056/13,906 ( 36.4%) | 10s | ~18s left | 488.2 rows/s | VRAM 525MB
  [title]  5,504/13,906 ( 39.6%) | 11s | ~17s left | 481.8 rows/s | VRAM 525MB
  [title]  6,016/13,906 ( 43.3%) | 13s | ~16s left | 479.2 rows/s | VRAM 525MB
  [title]  6,528/13,9

In [18]:
# === Xử lý CONTENT trên GPU ===
# Content dài hơn → batch_sz nhỏ hơn để tránh OOM trên 8GB VRAM
torch.cuda.empty_cache()
print("=" * 60)
print("PROCESSING CONTENT (GPU batch inference)")
print("=" * 60)
t0 = time.time()
df["content_bio"] = process_column_gpu(df["content"], "content", batch_sz=16)
content_time = time.time() - t0

print(f"\nContent done: {content_time:.1f}s")
print(f"\n{'='*60}")
print(f"TỔNG THỜI GIAN: {title_time + content_time:.1f}s")
print(f"  Title:   {title_time:.1f}s")
print(f"  Content: {content_time:.1f}s")
print(f"  VRAM peak: {torch.cuda.max_memory_allocated(0)/1024**2:.0f} MB")

PROCESSING CONTENT (GPU batch inference)
  [content]    512/13,906 (  3.7%) | 43s | ~1123s left | 11.9 rows/s | VRAM 525MB
  [content]  1,008/13,906 (  7.2%) | 79s | ~1008s left | 12.8 rows/s | VRAM 525MB
  [content]  1,504/13,906 ( 10.8%) | 119s | ~984s left | 12.6 rows/s | VRAM 525MB
  [content]  2,000/13,906 ( 14.4%) | 160s | ~950s left | 12.5 rows/s | VRAM 525MB
  [content]  2,512/13,906 ( 18.1%) | 201s | ~912s left | 12.5 rows/s | VRAM 525MB
  [content]  3,008/13,906 ( 21.6%) | 241s | ~871s left | 12.5 rows/s | VRAM 525MB
  [content]  3,504/13,906 ( 25.2%) | 279s | ~828s left | 12.6 rows/s | VRAM 525MB
  [content]  4,000/13,906 ( 28.8%) | 319s | ~789s left | 12.6 rows/s | VRAM 525MB
  [content]  4,512/13,906 ( 32.4%) | 364s | ~758s left | 12.4 rows/s | VRAM 525MB
  [content]  5,008/13,906 ( 36.0%) | 409s | ~727s left | 12.2 rows/s | VRAM 525MB
  [content]  5,504/13,906 ( 39.6%) | 453s | ~691s left | 12.2 rows/s | VRAM 525MB
  [content]  6,000/13,906 ( 43.1%) | 496s | ~654s left | 

## Thống kê phân bố Entity

In [19]:
def count_entities(bio_col: pd.Series) -> Counter:
    """Đếm B-tag cho mỗi entity type."""
    counter = Counter()
    for bio_str in bio_col:
        if not isinstance(bio_str, str):
            continue
        for line in bio_str.split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2 and parts[1].startswith("B-"):
                counter[parts[1][2:]] += 1
    return counter


def count_all_labels(bio_col: pd.Series) -> Counter:
    """Đếm tất cả labels."""
    counter = Counter()
    for bio_str in bio_col:
        if not isinstance(bio_str, str):
            continue
        for line in bio_str.split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2:
                counter[parts[1]] += 1
    return counter


for col_name, bio_col in [("TITLE", df["title_bio"]), ("CONTENT", df["content_bio"])]:
    print(f"\n{'='*55}")
    print(f" ENTITY DISTRIBUTION — {col_name}")
    print(f"{'='*55}")

    entities = count_entities(bio_col)
    total_ent = sum(entities.values())

    for etype, cnt in entities.most_common():
        pct = cnt / total_ent * 100 if total_ent > 0 else 0
        bar = "█" * int(pct / 2)
        print(f"  {etype:15s} {cnt:>8,}  ({pct:5.1f}%) {bar}")

    print(f"  {'─'*45}")
    print(f"  {'TOTAL':15s} {total_ent:>8,}")

    # Token distribution
    labels = count_all_labels(bio_col)
    total_tokens = sum(labels.values())
    o_count = labels.get("O", 0)
    entity_tokens = total_tokens - o_count
    print(f"\n  Total tokens:  {total_tokens:>10,}")
    print(f"  O tokens:      {o_count:>10,}  ({o_count/total_tokens*100:.1f}%)")
    print(f"  Entity tokens: {entity_tokens:>10,}  ({entity_tokens/total_tokens*100:.1f}%)")


 ENTITY DISTRIBUTION — TITLE
  LOCATION           4,891  ( 27.6%) █████████████
  ORGANIZATION       4,873  ( 27.5%) █████████████
  INDUSTRY           2,495  ( 14.1%) ███████
  MONEY              1,656  (  9.3%) ████
  PRODUCT            1,293  (  7.3%) ███
  DATE               1,155  (  6.5%) ███
  PERSON               977  (  5.5%) ██
  PERCENT              341  (  1.9%) 
  EVENT                 56  (  0.3%) 
  ─────────────────────────────────────────────
  TOTAL             17,737

  Total tokens:     212,936
  O tokens:         181,149  (85.1%)
  Entity tokens:     31,787  (14.9%)

 ENTITY DISTRIBUTION — CONTENT
  ORGANIZATION     165,399  ( 28.8%) ██████████████
  LOCATION         111,260  ( 19.3%) █████████
  INDUSTRY          91,724  ( 15.9%) ███████
  PERSON            50,411  (  8.8%) ████
  DATE              48,848  (  8.5%) ████
  MONEY             48,530  (  8.4%) ████
  PERCENT           34,348  (  6.0%) ██
  PRODUCT           24,154  (  4.2%) ██
  EVENT                

## Xuất dữ liệu NER

In [26]:
# === FORMAT 1: CoNLL ===

def write_conll(bio_col: pd.Series, path: str, append: bool = False) -> int:
    count = 0
    mode = "a" if append else "w"
    with open(path, mode, encoding="utf-8") as f:
        for bio_str in bio_col:
            if not isinstance(bio_str, str) or not bio_str.strip():
                continue
            for line in bio_str.strip().split("\n"):
                parts = line.rsplit(" ", 1)
                if len(parts) == 2:
                    f.write(f"{parts[0]}\t{parts[1]}\n")
            f.write("\n")
            count += 1
    return count

n1 = write_conll(df["title_bio"], "ner_conll.txt")
n2 = write_conll(df["content_bio"], "ner_conll.txt", append=True)
print(f"CoNLL: ner_conll.txt ({n1 + n2:,} sentences — {n1:,} title + {n2:,} content)")

CoNLL: ner_conll.txt (27,812 sentences — 13,906 title + 13,906 content)


In [27]:
# === FORMAT 2: JSON (HuggingFace-compatible) ===

def bio_str_to_record(row_id, text, bio_str, source_field="title"):
    tokens, labels = [], []
    if isinstance(bio_str, str) and bio_str.strip():
        for line in bio_str.strip().split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2:
                tokens.append(parts[0])
                labels.append(parts[1])
    return {"id": int(row_id), "source_field": source_field,
            "text": str(text), "tokens": tokens, "ner_tags": labels}


records = []
for _, r in df.iterrows():
    records.append(bio_str_to_record(r["id"], r["title"], r["title_bio"], "title"))
    records.append(bio_str_to_record(r["id"], r["content"], r["content_bio"], "content"))

with open("ner_labeled.json", "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"JSON: ner_labeled.json ({len(records):,} records — title + content)")

JSON: ner_labeled.json (27,812 records — title + content)


In [25]:
# === FORMAT 3: CSV ===

df.to_csv("ner_labeled_data.csv", index=False, encoding="utf-8-sig")
print(f"CSV: ner_labeled_data.csv ({len(df):,} rows, {list(df.columns)})")

CSV: ner_labeled_data.csv (13,906 rows, ['id', 'title', 'content', 'source', 'date', 'title_bio', 'content_bio'])


## Quality Check

Kiểm tra mẫu ngẫu nhiên + thống kê coverage.

In [23]:
import random
random.seed(42)

def extract_entities_from_bio(bio_str: str) -> List[Tuple[str, str]]:
    """Trích entity text từ BIO string."""
    entities = []
    current_tokens = []
    current_type = None

    for line in str(bio_str).strip().split("\n"):
        parts = line.rsplit(" ", 1)
        if len(parts) != 2:
            continue
        token, label = parts

        if label.startswith("B-"):
            if current_tokens and current_type:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = [token]
            current_type = label[2:]
        elif label.startswith("I-") and current_type:
            current_tokens.append(token)
        else:
            if current_tokens and current_type:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = []
            current_type = None

    if current_tokens and current_type:
        entities.append((" ".join(current_tokens), current_type))

    return entities


# Random sample check
sample_ids = random.sample(range(len(df)), min(8, len(df)))

for idx in sample_ids:
    row = df.iloc[idx]
    title = str(row["title"])

    print(f"{'─'*60}")
    print(f"ID {row['id']}: {title[:75]}{'...' if len(title)>75 else ''}")

    entities = extract_entities_from_bio(row["title_bio"])
    if entities:
        for text, etype in entities:
            print(f"  ● {etype:15s} → {text}")
    else:
        print("  (no entities)")

print(f"\n{'═'*60}")
print("COVERAGE STATS")
print(f"{'═'*60}")

no_entity = df["title_bio"].apply(
    lambda x: not any(
        line.rsplit(" ", 1)[-1].startswith("B-")
        for line in str(x).strip().split("\n")
        if line.strip()
    )
)
n_empty = no_entity.sum()
print(f"Titles without any entity: {n_empty:,} / {len(df):,} ({n_empty/len(df)*100:.1f}%)")
print(f"Titles with ≥1 entity:    {len(df)-n_empty:,} / {len(df):,} ({(len(df)-n_empty)/len(df)*100:.1f}%)")

if n_empty > 0:
    print(f"\nSample titles without entities:")
    for _, r in df[no_entity].head(5).iterrows():
        print(f"  ID {r['id']}: {str(r['title'])[:75]}")

────────────────────────────────────────────────────────────
ID 10477: 'Cơn sốt' mở cửa hàng Mixue ở Việt Nam hạ nhiệt, la liệt tin rao nhượng lại...
  ● LOCATION        → Việt Nam
────────────────────────────────────────────────────────────
ID 1825: Gia hạn nhận hồ sơ tham gia Giải thưởng Chuyển đổi số Việt Nam 2025
  (no entities)
────────────────────────────────────────────────────────────
ID 410: Diễn biến mới của cổ phiếu Hóa chất Đức Giang trong phiên VN-Index tăng 47 ...
  (no entities)
────────────────────────────────────────────────────────────
ID 12150: Mô hình AI mới cho phép tấn công máy tính từ xa thông qua bức xạ điện từ
  (no entities)
────────────────────────────────────────────────────────────
ID 4507: Giá vàng tăng nóng, 'cá mập' còn mạnh tay gom vàng?
  (no entities)
────────────────────────────────────────────────────────────
ID 4013: Biến động kinh tế, gửi niềm tin vào vàng, trái phiếu hay USD?
  (no entities)
───────────────────────────────────────────────────────

## Xem chi tiết BIO output mẫu

In [24]:
# Hiển thị full BIO output cho 3 bài đầu
for i in range(min(3, len(df))):
    row = df.iloc[i]
    title = str(row["title"])
    bio_str = row["title_bio"]

    print(f"{'═'*60}")
    print(f"BÀI {row['id']}: {title[:70]}...")
    print(f"{'═'*60}")

    if isinstance(bio_str, str) and bio_str.strip():
        for line in bio_str.split("\n"):
            parts = line.rsplit(" ", 1)
            if len(parts) == 2:
                token, label = parts
                marker = " ◀" if label != "O" else ""
                print(f"  {token:30s} {label}{marker}")
    print()

════════════════════════════════════════════════════════════
BÀI 1: Motorola chuẩn bị ra mắt phiên bản điện thoại nắp gập Razr giá rẻ 12 t...
════════════════════════════════════════════════════════════
  Motorola                       B-ORGANIZATION ◀
  chuẩn                          O
  bị                             O
  ra                             O
  mắt                            O
  phiên                          O
  bản                            O
  điện                           O
  thoại                          O
  nắp                            O
  gập                            O
  Razr                           B-PRODUCT ◀
  giá                            O
  rẻ                             O
  12                             B-MONEY ◀
  triệu                          I-MONEY ◀
  đồng                           I-MONEY ◀

════════════════════════════════════════════════════════════
BÀI 2: 2 tỷ phú quốc tế 'xông đất', hứa đưa Bình Định thành trung tâm du lịch...
══════════

## Tổng kết

### Kiến trúc Hybrid NER Pipeline (GPU-accelerated)

| Layer | Nguồn | Entity Types | Ưu điểm |
|-------|-------|-------------|---------|
| **1. Electra Model** | `NlpHUST/ner-vietnamese-electra-base` | PERSON, ORGANIZATION, LOCATION | Hiểu ngữ cảnh, F1=0.92, **batch inference trên GPU** |
| **2. Regex** | Regular expressions | MONEY, DATE, PERCENT | Chính xác cao cho entity có cấu trúc, **boundary-safe** |
| **3. Gazetteer** | Từ điển thủ công | PRODUCT, EVENT, INDUSTRY + bổ sung | **Pre-sorted** (dài→ngắn), cover entity model bỏ sót |

### Preprocessing
- **Text Normalization:** Unicode NFC, smart quotes → ASCII, NBSP/ZWSP → loại, multi-space → gộp
- **Gazetteer pre-sort:** 1 lần sort theo length giảm dần → match dài nhất trước (e.g. "iPhone 17" trước "iPhone")
- **Regex boundary:** `\b` + lookahead/lookbehind cho tất cả patterns, deduplicate overlapping matches
- **Vietnamese boundary:** Custom `_is_word_char()` hỗ trợ dấu tiếng Việt (ăâêôơưđ...) cho gazetteer matching

### GPU Acceleration
- **Model:** chạy batch inference trên RTX 3070 8GB
- **Title:** batch_size=64 (text ngắn, ~15 tokens)
- **Content:** batch_size=16 (text dài, ~500+ tokens, tránh OOM)
- Tốc độ ước tính: **~5-10x nhanh hơn CPU**

### Chiến lược Merge
- **Priority:** Model > Regex > Gazetteer
- **Overlap:** Span ưu tiên cao hơn thắng; cùng priority → span dài hơn thắng
- **Gazetteer occupied set:** Span dài match trước → span ngắn không đè lên (e.g. "iPhone 17" blocks "iPhone")

### Output Files
| File | Format | Nội dung |
|------|--------|---------|
| `ner_conll.txt` | CoNLL | BIO labels (title + content gộp) |
| `ner_labeled.json` | JSON | tokens + ner_tags (HuggingFace ready, title + content) |
| `ner_labeled_data.csv` | CSV | Full data + title_bio + content_bio |